# تدريب النموذج العربي ونشر الموقع — Train the Arabic model & publish the website

**قبل البدء / Before you start:** Runtime → Change runtime type → **GPU (T4)**. ثم شغّل الخلايا بالترتيب.

> تنبؤات بحثية فقط — ليست نصيحة طبية. *Research predictions only — not medical advice.*

What this notebook does (nothing is faked; every number comes from real training):
1. Downloads the code and the public AraFacts dataset (CC BY-NC 4.0, SHA-256 verified).
2. Asks **you** to approve the label mapping (your name is recorded).
3. Builds a verified, group-aware **2-class** split (SUPPORTED vs MISLEADING); NEEDS_REVIEW comes from calibrated abstention.
4. Fine-tunes `aubmindlab/bert-base-arabertv02`, calibrates on validation, evaluates once on test.
5. Optionally publishes the website + trained model to your free Hugging Face Space (public link).

In [ ]:
#@title 1) Get the code / تنزيل الكود
REPO = "https://github.com/Amer-omran/biobank.git"   #@param {type:"string"}
BRANCH = "claude/quirky-tesla-2a5zhb"                 #@param {type:"string"}
import getpass, os, subprocess
url = REPO
if input("Is the GitHub repository private? (y/N) ").strip().lower() == "y":
    tok = getpass.getpass("GitHub token (read access): ")
    url = REPO.replace("https://", f"https://{tok}@")
if not os.path.exists("/content/biobank"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "/content/biobank"], check=True)
%cd /content/biobank/health-misinfo
!git log -1 --format='%h %s'

In [ ]:
#@title 2) Install dependencies / تثبيت المتطلبات (Colab already has PyTorch)
!pip -q install -r backend/requirements.txt "transformers==5.24.0" "accelerate==1.16.0" "safetensors==0.8.0" matplotlib huggingface_hub
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable a GPU runtime!")

In [ ]:
#@title 3) Download AraFacts and import the health claims / تنزيل البيانات واستيرادها
%env HMD_DATABASE_URL=sqlite:////content/hmd.db
%env HMD_ENVIRONMENT=development
!bash scripts/fetch_datasets.sh
%cd backend
!python -m healthmisinfo.cli init-db
!python -m healthmisinfo.cli import-arafacts --file "../data/raw/arafacts/Dataset/AraFacts 2/AraFacts 2.csv" --name arafacts-health --version v2-2024-06-23 | tail -n 5

## 4) اعتماد مواءمة التصنيفات — YOUR decision

Review the table, edit `DECISIONS` if you disagree (values: `SUPPORTED`, `MISLEADING`, `NEEDS_REVIEW`, `EXCLUDE`),
write your name and set `I_CONFIRM = True`. Defaults = the proposals in the repository, except the two
framing-only labels (`انحياز`, `فيديو دعائي`) which are set to `EXCLUDE` as recommended.
راجع الجدول، وعدّل القرارات إن لزم، واكتب اسمك، ثم اجعل `I_CONFIRM = True`.

In [ ]:
import json, pandas as pd
m = json.load(open("../data/registry/label_mappings/arafacts.json"))
DECISIONS = {e["original"]: e["proposed_label"] for e in m["entries"]}
DECISIONS.update({"انحياز": "EXCLUDE", "فيديو دعائي": "EXCLUDE"})   # recommended overrides
pd.set_option("display.max_rows", 100)
display(pd.DataFrame([{"original": e["original"], "decision": DECISIONS[e["original"]], "ambiguity": e["ambiguity"], "rationale": e["rationale"]} for e in m["entries"]]))

In [ ]:
APPROVER_NAME = ""     #@param {type:"string"}
I_CONFIRM = False      #@param {type:"boolean"}
assert APPROVER_NAME.strip() and I_CONFIRM, "Write your name and tick I_CONFIRM to approve the mapping."
import subprocess
by_target = {}
for orig, tgt in DECISIONS.items():
    by_target.setdefault(tgt, []).append(orig)
for tgt, origs in by_target.items():
    subprocess.run(["python", "-m", "healthmisinfo.cli", "approve-mapping", "--source", "arafacts",
                    "--approved-by", APPROVER_NAME, "--entries", ",".join(origs), "--set-label", tgt], check=True)
!python -m healthmisinfo.cli refresh-dataset --id 1 | python -c "import json,sys; d=json.load(sys.stdin); print('eligible:', d['eligible'], d['eligible_label_distribution'])"

In [ ]:
#@title 5) Build the verified 2-class split / إنشاء التقسيم
!python -m healthmisinfo.cli build-splits --language ar --version v1-binary --seed 42 --label-scheme binary | python -c "import json,sys; r=json.load(sys.stdin); print(r['status'], r['split_sizes'], r['label_distribution'])"

In [ ]:
#@title 6) Fine-tune AraBERT (≈15–30 min on a T4) / التدريب
!python -m healthmisinfo.ml.train --language ar --split-dir ../data/processed/ar/v1-binary --seed 42 --epochs 4

In [ ]:
#@title 7) Results (measured on the held-out test split) / النتائج
import glob, json
from IPython.display import Markdown, Image
run = sorted(glob.glob("../models/ar/*/evaluation.json"))[-1].rsplit("/", 1)[0]
ev = json.load(open(f"{run}/evaluation.json"))
print("test:", {k: round(ev['test'][k], 4) for k in ['accuracy', 'macro_precision', 'macro_recall', 'macro_f1']}, "n =", ev['test']['n'])
display(Image(f"{run}/confusion_matrix_test.png"))
display(Markdown(open(f"{run}/model_card.md").read()))

In [ ]:
#@title 8) Save everything to your computer / حفظ النتائج
!cd .. && zip -qr /content/results.zip models data/processed data/registry/label_mappings/arafacts.json
from google.colab import files; files.download("/content/results.zip")
print("Also commit data/registry/label_mappings/arafacts.json to GitHub: it records your approval.")

## 9) النشر على رابط عام مجاني — Publish to a public link (Hugging Face Space)

1. Create a free account at https://huggingface.co and a **write** token at https://huggingface.co/settings/tokens
2. Run the cell. It creates `<your-username>/health-claims-ai`, uploads the website **and the trained model**, and prints the link.
3. The first build takes about 10–15 minutes. Open the printed `https://...hf.space` link and sign in with the admin e-mail/password you enter here.

In [ ]:
HF_USERNAME = ""                     #@param {type:"string"}
SPACE_NAME = "health-claims-ai"      #@param {type:"string"}
ADMIN_EMAIL = ""                     #@param {type:"string"}
import getpass, os, subprocess
os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face WRITE token: ")
os.environ["HMD_ADMIN_PASSWORD"] = getpass.getpass("Choose the website admin password (min 10 chars, mix letters/digits/symbols): ")
subprocess.run(["python", "../scripts/deploy_hf_space.py", "--space", f"{HF_USERNAME}/{SPACE_NAME}",
                "--admin-email", ADMIN_EMAIL, "--models-dir", "../models"], check=True)